# 2.1 — Conditional Generation & Classifier-Free Guidance

**Mechanism of the day:** tell the generator what you want — and then turn a knob that
controls how *insistently* it listens.

Track 1 built five engines that sample `p(x)`. Not one of them takes an instruction.
That is the gap Track 2 closes, and this is the first and most widely used answer.

**Step one: conditioning.** Feed the label `y` in as an extra input and train
`v_theta(x, t, y)` instead of `v_theta(x, t)`. That is barely a change — an embedding
added to the hidden state — and it gives you `p(x | y)`.

**Step two: guidance.** Conditioning alone is often too polite. The model has learned
`p(x | y)` faithfully, including all the cases where the label is ambiguous, so samples
drift toward whatever is generically likely. You frequently want something *more*
`y`-like than the data average.

The trick, **classifier-free guidance**, is beautiful:

1. During training, randomly replace `y` with a **null token** ~15% of the time. One
   network now learns both the conditional and the unconditional field.
2. During sampling, run it twice and **extrapolate along the difference**:

```
v_guided = v_uncond  +  w * (v_cond - v_uncond)
```

Read that geometrically. `v_cond - v_uncond` is the direction that *makes a sample more
like class `y`*. Guidance says: don't just take one step of it — take `w` of them.

- `w = 0` -> pure unconditional. The label is ignored entirely.
- `w = 1` -> ordinary conditional sampling.
- `w > 1` -> **extrapolation past what the model believes**, exaggerating the condition.

The name says what it does not need: no separate classifier, no gradients through an
external model (that is 2.2). Just one network, two forward passes, and a subtraction.

**Our test bed** is the Ramachandran plot from 1.3/1.4, with one honest change: we add
the **PPII** basin, which genuinely overlaps the beta region in real proteins. That
overlap matters — with well-separated classes, plain conditioning already scores 100%
and guidance has nothing to show. With a genuinely ambiguous label, you can *measure*
what guidance buys, and what it costs.

**How to use this notebook:** implement the reps, make the checkpoints pass. Solutions
at the bottom. Trains in ~12s on a laptop CPU.

In [ ]:
import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
INK = '#52514e'
CLS_COLORS = ['#2a78d6', '#008300', '#e87ba4', '#eda100']    # 4 distinct hues

# Ramachandran with FOUR regions. beta and PPII overlap, exactly as in real proteins —
# that ambiguity is what makes guidance measurable.
BASINS = [('alpha-helix', (-63.0, -43.0),  (25.0, 25.0), 0.40),
          ('beta-sheet',  (-125.0, 130.0), (28.0, 26.0), 0.30),
          ('PPII',        (-70.0, 145.0),  (20.0, 20.0), 0.22),
          ('left-handed', (57.0, 40.0),    (18.0, 18.0), 0.08)]
SCALE = 180.0
NCLS = len(BASINS)
NULL = NCLS                      # the "no condition" token -> index 4

def make_data(n):
    w = np.array([b[3] for b in BASINS])
    which = rng.choice(NCLS, size=n, p=w)
    out = np.zeros((n, 2))
    for i, (_, mu, sd, _) in enumerate(BASINS):
        m = which == i; k = int(m.sum())
        out[m, 0] = rng.normal(mu[0], sd[0], k)
        out[m, 1] = rng.normal(mu[1], sd[1], k)
    return out.astype(np.float32), which

raw, lab = make_data(8000)
X0 = torch.from_numpy(raw / SCALE)
Y0 = torch.from_numpy(lab).long()
CEN = np.array([b[1] for b in BASINS])

def assign(pts_deg):
    '''Nearest basin centre for each point.'''
    return ((pts_deg[:, None, :] - CEN[None]) ** 2).sum(-1).argmin(1)

fig, ax = plt.subplots(figsize=(4.6, 4.4))
for i, (name, mu, _, w) in enumerate(BASINS):
    m = lab == i
    ax.scatter(raw[m, 0], raw[m, 1], s=4, alpha=.3, lw=0, color=CLS_COLORS[i],
               label='%s (%.0f%%)' % (name, 100 * w))
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
ax.set_xlabel('phi'); ax.set_ylabel('psi'); ax.grid(alpha=.15)
ax.set_title('four conditions — beta and PPII overlap')
ax.legend(frameon=False, fontsize=8, markerscale=2, loc='lower right')
plt.show()
print('class priors:', np.round(np.bincount(lab) / len(lab), 3))
print('NULL token index =', NULL)

## Part 1 — training one network to be two models

The whole method rests on a single training-time trick: sometimes hide the label. The
network cannot tell in advance whether it will be given a real condition or the null
token, so it must learn **both** `v(x, t, y)` and `v(x, t, NULL)` — the conditional and
the unconditional velocity field — inside one set of weights.

Choosing the drop rate is a real tradeoff: too low and the unconditional branch is
undertrained (guidance becomes noisy); too high and you waste capacity. ~10-20% is
standard, and we use 15%.

### Rep 1 — `drop_labels(y, p=0.15)`
Return a copy of `y` with each entry independently replaced by `NULL` with probability
`p`. Do **not** modify `y` in place — it is the training data.

In [ ]:
def drop_labels(y, p=0.15):
    '''Randomly replace labels with the NULL token.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
y = torch.randint(0, NCLS, (20000,))
yd = drop_labels(y, 0.15)
frac = (yd == NULL).float().mean().item()
assert abs(frac - 0.15) < 0.02, 'about 15%% should be dropped, got %.3f' % frac
assert (y < NCLS).all(), 'the original labels must not be modified in place'
kept = yd != NULL
assert (yd[kept] == y[kept]).all(), 'surviving labels must be unchanged'
assert (drop_labels(y, 0.0) == y).all() and (drop_labels(y, 1.0) == NULL).all()
print('dropped %.1f%% of labels to NULL ✓  (one net, two models)' % (100 * frac))

In [ ]:
def timestep_embedding(t, dim=64):
    half = dim // 2
    fr = torch.exp(-math.log(10000) * torch.arange(half, dtype=torch.float32) / half)
    a = t[:, None].float() * fr[None]
    return torch.cat([torch.cos(a), torch.sin(a)], -1)

class CondNet(nn.Module):
    '''1.4's velocity network plus a label embedding. That is the entire change.'''
    def __init__(self, d=128, tdim=64):
        super().__init__(); self.tdim = tdim
        self.tmlp = nn.Sequential(nn.Linear(tdim, d), nn.SiLU(), nn.Linear(d, d))
        self.yemb = nn.Embedding(NCLS + 1, d)          # +1 row for NULL
        self.inp = nn.Linear(2, d)
        self.h1 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.h2 = nn.Sequential(nn.SiLU(), nn.Linear(d, d))
        self.out = nn.Sequential(nn.SiLU(), nn.Linear(d, 2))
    def forward(self, x, t, y):
        h = self.inp(x) + self.tmlp(timestep_embedding(t, self.tdim)) + self.yemb(y)
        h = h + self.h1(h); h = h + self.h2(h)
        return self.out(h)

model = CondNet()
print('parameters:', sum(p.numel() for p in model.parameters()))

### Rep 2 — `cfg_loss(model, x0, y, p_drop=0.15)`
1.4's flow-matching loss with two additions: pass the (dropped) labels to the model.
Same straight path, same constant velocity target `x1 - x0`.

In [ ]:
def cfg_loss(model, x0, y, p_drop=0.15):
    '''Conditional flow matching with label dropout.'''
    # YOUR CODE HERE
    # hint: x1 = torch.randn_like(x0); t = torch.rand(len(x0))
    #       xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    #       predict with model(xt, t * 1000, drop_labels(y, p_drop)), target x1 - x0
    raise NotImplementedError

# --- checkpoint ---
l = cfg_loss(model, X0[:512], Y0[:512])
assert l.dim() == 0 and l.requires_grad
assert 0.3 < l.item() < 3.0, 'untrained loss should be order 1'
print('untrained conditional loss %.3f ✓' % l.item())

In [ ]:
opt = torch.optim.AdamW(model.parameters(), lr=2e-3)
hist, t0 = [], time.time()
for step in range(1, 6001):
    ix = torch.randint(0, len(X0), (256,))
    loss = cfg_loss(model, X0[ix], Y0[ix])
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 200 == 0 or step == 1:
        hist.append((step, loss.item()))
        if step % 2000 == 0 or step == 1:
            print('step %4d  loss %.4f' % (step, loss.item()))
print('trained in %.1fs on CPU' % (time.time() - t0))

h = np.array(hist)
fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.plot(h[:, 0], h[:, 1], color=CLS_COLORS[0], lw=2)
ax.set_xlabel('step'); ax.set_ylabel('MSE on velocity'); ax.grid(alpha=.15)
ax.set_title('one network, conditional and unconditional at once')
plt.show()

## Part 2 — guidance

Two forward passes, one subtraction, one multiply-add. This is the cell that matters.

```
v_guided = v_uncond + w * (v_cond - v_uncond)
```

Equivalently `v_guided = (1 - w) * v_uncond + w * v_cond`, which makes the structure
obvious: it is a **linear extrapolation** between the two fields. At `w = 1` you land
exactly on the conditional. Past that you keep going in the same direction — beyond
anything the model was ever trained to produce. That is why large `w` sharpens the
condition *and* distorts the distribution: you are extrapolating, not interpolating.

(Small efficiency note the real implementations use: batch the conditional and
unconditional inputs together so it is one forward pass over `2n` rows rather than two
passes over `n`. We keep them separate here for clarity.)

### Rep 3 — `guided_velocity(model, x, t, y, w)`
Evaluate the model with the real labels `y` and again with `NULL`, then combine.
`t` is a python float; the network wants `t * 1000` as a `[n]` tensor.

In [ ]:
@torch.no_grad()
def guided_velocity(model, x, t, y, w):
    '''v_uncond + w * (v_cond - v_uncond).'''
    # YOUR CODE HERE
    # hint: tt = torch.full((len(x),), t * 1000)
    #       null = torch.full((len(x),), NULL, dtype=torch.long)
    raise NotImplementedError

# --- checkpoint ---
xs = torch.randn(256, 2)
ys = torch.zeros(256, dtype=torch.long)
v_c = guided_velocity(model, xs, 0.5, ys, 1.0)
v_u = guided_velocity(model, xs, 0.5, ys, 0.0)
assert v_c.shape == xs.shape
assert not torch.allclose(v_c, v_u), 'conditional and unconditional must differ'
# w=2 must be exactly one more step of the same difference
v_2 = guided_velocity(model, xs, 0.5, ys, 2.0)
assert torch.allclose(v_2, v_u + 2 * (v_c - v_u), atol=1e-5), 'check the formula'
assert torch.allclose(v_2 - v_c, v_c - v_u, atol=1e-5), 'guidance is linear extrapolation'
print('guidance ✓ — w=0 gives unconditional, w=1 conditional, w>1 extrapolates')

### Rep 4 — `sample_cfg(model, n, cls, w=1.0, steps=32)`
1.4's Euler ODE loop, with `guided_velocity` in place of the plain model call.
Every sample is conditioned on the single class `cls`.

In [ ]:
@torch.no_grad()
def sample_cfg(model, n, cls, w=1.0, steps=32):
    '''Generate n points of class cls at guidance scale w.'''
    # YOUR CODE HERE
    # hint: x = torch.randn(n, 2); y = torch.full((n,), cls, dtype=torch.long)
    #       dt = 1/steps; t goes 1 -> 0; x = x - dt * guided_velocity(...)
    raise NotImplementedError

# --- checkpoint ---
p = sample_cfg(model, 500, 0, w=1.0).numpy() * SCALE
assert p.shape == (500, 2)
assert (assign(p) == 0).mean() > 0.8, 'asking for alpha should mostly give alpha'
print('conditional sampling ✓ — %.1f%% of "alpha-helix" requests landed in that basin'
      % (100 * (assign(p) == 0).mean()))

## Part 3 — what guidance buys, and what it costs

Two numbers per guidance scale:

- **adherence** — the fraction of samples that land in the basin you asked for,
- **spread** — the average within-basin standard deviation, in degrees. This is our
  diversity measure, and crucially we know the target: the **real data's** spread.

Watch three regimes. At `w = 0` the per-class hit rates should collapse onto the class
**priors** — the model is ignoring you completely. At `w = 1` adherence jumps but the
ambiguous `beta` class lags behind the others. Past `w = 1` adherence saturates at 100%
while the spread falls *below* the real data — you are buying obedience with diversity.

Watch the spread curve past its minimum, though: it turns around and grows again. That
is not diversity coming back, it is **distortion**. At large `w` you are extrapolating
so far outside the trained region that samples get flung into distorted configurations,
which widens the spread while making the samples *worse*. A metric moving in the
"good" direction for a bad reason — always check what the samples actually look like.

### Rep 5 — `basin_accuracy(pts_deg, cls)`
Fraction of points whose nearest basin centre is `cls`. Use the provided `assign`.

In [ ]:
def basin_accuracy(pts_deg, cls):
    '''Fraction of points landing in the requested basin.'''
    # YOUR CODE HERE
    raise NotImplementedError

# --- checkpoint ---
assert abs(basin_accuracy(raw[lab == 2], 2) - 1.0) < 0.35, 'real PPII points sit near PPII'
assert basin_accuracy(raw, 0) < 0.6, 'the whole dataset is not one class'
WS = [0.0, 0.5, 1.0, 2.0, 3.0, 5.0, 8.0]
acc, spr = [], []
for w in WS:
    a, s = [], []
    for c in range(NCLS):
        p = sample_cfg(model, 1200, c, w=w).numpy() * SCALE
        a.append(basin_accuracy(p, c))
        sel = p[assign(p) == c]
        if len(sel) > 20:
            s.append(sel.std(0).mean())
    acc.append(np.mean(a)); spr.append(np.mean(s))
    print('w=%4.1f  ' % w + '  '.join('%.3f' % v for v in a) + '   | mean %.3f  spread %.1f deg'
          % (np.mean(a), np.mean(s)))

REAL_SPREAD = float(np.mean([raw[lab == i].std(0).mean() for i in range(NCLS)]))
priors = np.bincount(lab) / len(lab)
print('\nclass priors            ', np.round(priors, 3))
print('real within-basin spread %.1f deg' % REAL_SPREAD)
print('\nAt w=0 the hit rates ARE the priors — the label is being ignored. ✓')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(WS, acc, '-o', color=CLS_COLORS[0], lw=2, ms=6)
axes[0].axhline(float(priors.mean()), color=INK, ls=':', lw=1.5)
axes[0].text(8, priors.mean(), 'ignoring the label', ha='right', va='bottom',
             fontsize=9, color=INK)
axes[0].set_xlabel('guidance scale w'); axes[0].set_ylabel('adherence (fraction correct)')
axes[0].set_title('guidance buys obedience', fontsize=10); axes[0].grid(alpha=.15)

axes[1].plot(WS, spr, '-o', color=CLS_COLORS[1], lw=2, ms=6)
axes[1].axhline(REAL_SPREAD, color=INK, ls='--', lw=1.5)
axes[1].text(8, REAL_SPREAD, 'real data', ha='right', va='bottom', fontsize=9, color=INK)
axes[1].set_xlabel('guidance scale w'); axes[1].set_ylabel('within-basin spread (deg)')
axes[1].set_title('and pays for it in diversity', fontsize=10); axes[1].grid(alpha=.15)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, w in zip(axes, [0.0, 1.0, 3.0, 8.0]):
    for c in range(NCLS):
        p = sample_cfg(model, 700, c, w=w).numpy() * SCALE
        ax.scatter(p[:, 0], p[:, 1], s=4, alpha=.3, lw=0, color=CLS_COLORS[c])
    ax.set_xlim(-180, 180); ax.set_ylim(-180, 180); ax.grid(alpha=.15)
    ax.set_xlabel('phi'); ax.set_title('w = %g' % w, fontsize=10)
axes[0].set_ylabel('psi')
plt.tight_layout(); plt.show()
print('Colour = the class REQUESTED. At w=0 the colours are scrambled across basins')
print('(the label did nothing); by w=3 each colour sits in its own basin, tightly. ✓')

## Reflection — what just transferred

- **Conditioning is cheap; guidance is the interesting part.** Adding `y` to the
  network is a one-line change. The leverage comes from having *both* fields and
  extrapolating between them.
- **The null token is the whole trick.** Dropping the label during training turns one
  network into two models, which is what makes `v_cond - v_uncond` computable at all.
- **`v_uncond + w (v_cond - v_uncond)` is a direction, scaled.** At `w > 1` you are
  extrapolating past anything the model was trained on — which is exactly why it
  sharpens adherence *and* distorts the distribution.
- **We measured the cost.** By `w = 2` our samples were tighter than the real data;
  by `w = 8` they were caricatures. In images this shows up as over-saturated,
  over-typical pictures; in protein design it shows up as sequences that ace the oracle
  and fail in the lab.
- **This is 0.2's frontier for the third time** — after tilting temperature and after
  decoding order. Reward up, diversity down, always. You now have three different
  mechanisms that produce the same curve, which is a good sign it is structural rather
  than an artifact.
- **Everything real uses this.** Stable Diffusion, Imagen, and the conditional protein
  generators all ship CFG, and "guidance scale" in any image tool is exactly the `w`
  you just implemented.

**Next rep:** `2.2 Classifier guidance & training-free guidance` — what if you cannot
retrain the generator? Take a **frozen** model and steer it with the gradient of a
separate property oracle. That is how you control a model someone else trained, and it
is the differentiable version of 0.2's reward tilting.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def drop_labels(y, p=0.15):
    y = y.clone()                                   # never modify the training labels
    y[torch.rand(len(y)) < p] = NULL
    return y

def cfg_loss(model, x0, y, p_drop=0.15):
    x1 = torch.randn_like(x0)
    t = torch.rand(len(x0))
    xt = (1 - t[:, None]) * x0 + t[:, None] * x1
    return F.mse_loss(model(xt, t * 1000, drop_labels(y, p_drop)), x1 - x0)

@torch.no_grad()
def guided_velocity(model, x, t, y, w):
    tt = torch.full((len(x),), t * 1000)
    v_cond = model(x, tt, y)
    if w == 1.0:
        return v_cond                               # skip the second pass when possible
    v_uncond = model(x, tt, torch.full((len(x),), NULL, dtype=torch.long))
    return v_uncond + w * (v_cond - v_uncond)

@torch.no_grad()
def sample_cfg(model, n, cls, w=1.0, steps=32):
    x = torch.randn(n, 2)
    y = torch.full((n,), cls, dtype=torch.long)
    dt = 1.0 / steps
    for i in range(steps):
        t = 1.0 - i * dt
        x = x - dt * guided_velocity(model, x, t, y, w)
    return x

def basin_accuracy(pts_deg, cls):
    return float((assign(pts_deg) == cls).mean())

print('reference solutions loaded — re-run the checkpoint cells above')